In [ ]:
import pandas as pd
import json
import re

In [4]:
file_path =  "arxiv-metadata-oai-snapshot.json"

target_categories = ('cs.','math.','stat.ML')

In [6]:
all_matching_papers = []

with open(file_path,'r') as f:
    for line in f:
        paper = json.loads(line)
        categories = paper.get('categories','')

        if any(target in categories for target in target_categories):
            all_matching_papers.append({
                'id': paper.get('id'),
                'title': paper.get('title').replace('\n', ' ').strip(),
                'abstract': paper.get('abstract').replace('\n', ' ').strip(),
                'categories': categories})
df_all = pd.DataFrame(all_matching_papers)
print(f"Found {len(df_all)} total papers in CS, Math, and ML.")


Found 1775575 total papers in CS, Math, and ML.


In [ ]:
def get_main_domain(cat_string):
    if 'stat.ML' in cat_string or 'cs.LG' in cat_string:
        return 'Machine Learning'
    elif 'cs.' in cat_string:
        return 'Computer Science'
    else:
        return 'Mathematics'
df_all['main_domain'] = df_all['categories'].apply(get_main_domain)

In [17]:
df_sampled = df_all.groupby('main_domain').sample(n=166666, random_state=42).reset_index(drop=True)
df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

In [18]:
df_sampled

,id,title,abstract,categories,main_domain
0,2511.06192,SoK: Systematizing a Decade of Architectural R...,"A decade after its academic introduction, RowH...",cs.CR cs.AR,Computer Science
1,1609.09481,Fast learning rates with heavy-tailed losses,We study fast learning rates when the losses a...,stat.ML cs.LG,Machine Learning
2,2111.06250,A flexible film thermocouple temperature sensor,This article introduces a thin-film thermocoup...,cond-mat.mtrl-sci physics.app-ph,Computer Science
3,1511.06620,Use of Eigenvector Centrality to Detect Graph ...,Graph Isomorphism is one of the classical prob...,cs.SI cs.DM cs.DS,Computer Science
4,1801.09557,Families of Solutions of Algebraic Riccati Equ...,We consider Homogeneous Algebraic Riccati Equa...,math.OC,Mathematics
...,...,...,...,...,...
499993,2106.07524,MIA-COV19D: COVID-19 Detection through 3-D Che...,Early and reliable COVID-19 diagnosis based on...,eess.IV cs.CV cs.LG,Machine Learning
499994,1607.07213,A transcendental approach to injectivity theor...,"In this paper, we study transcendental aspects...",math.CV math.AG math.DG,Mathematics
499995,1512.08723,Emergent Weyl excitations in systems of polar ...,Weyl fermions are massless chiral particles fi...,cond-mat.quant-gas cond-mat.mes-hall cond-mat....,Computer Science
499996,2202.11885,A Partition-and-Merge Algorithm for Solving th...,The Steiner tree problem aims to determine a m...,cs.DS,Computer Science


In [19]:
def clean_latex(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Replace all math equations (anything between $...$) with a standard token
    text = re.sub(r'\$.*?\$', ' math_formula ', text)
    
    # 2. Remove LaTeX macros and commands (e.g., \emph, \textbf, \alpha, \frac)
    text = re.sub(r'\\[a-zA-Z]+', ' ', text)
    
    # 3. Remove stray curly braces { } left behind by LaTeX groupings
    text = re.sub(r'[{}]', ' ', text)
    
    # 4. Clean up any double spaces created by removing the above elements
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Cleaning LaTeX from abstracts and titles...")

# Apply the function to create clean columns for your TF-IDF vectorizer
df_sampled['abstract_clean'] = df_sampled['abstract'].apply(clean_latex)
df_sampled['title_clean'] = df_sampled['title'].apply(clean_latex)

print("Cleaning complete!")
print(df_sampled[['abstract', 'abstract_clean']].head())

Cleaning LaTeX from abstracts and titles...
Cleaning complete!
                                            abstract  \
0  A decade after its academic introduction, RowH...   
1  We study fast learning rates when the losses a...   
2  This article introduces a thin-film thermocoup...   
3  Graph Isomorphism is one of the classical prob...   
4  We consider Homogeneous Algebraic Riccati Equa...   

                                      abstract_clean  
0  A decade after its academic introduction, RowH...  
1  We study fast learning rates when the losses a...  
2  This article introduces a thin-film thermocoup...  
3  Graph Isomorphism is one of the classical prob...  
4  We consider Homogeneous Algebraic Riccati Equa...  


In [20]:
df_sampled

,id,title,abstract,categories,main_domain,abstract_clean,title_clean
0,2511.06192,SoK: Systematizing a Decade of Architectural R...,"A decade after its academic introduction, RowH...",cs.CR cs.AR,Computer Science,"A decade after its academic introduction, RowH...",SoK: Systematizing a Decade of Architectural R...
1,1609.09481,Fast learning rates with heavy-tailed losses,We study fast learning rates when the losses a...,stat.ML cs.LG,Machine Learning,We study fast learning rates when the losses a...,Fast learning rates with heavy-tailed losses
2,2111.06250,A flexible film thermocouple temperature sensor,This article introduces a thin-film thermocoup...,cond-mat.mtrl-sci physics.app-ph,Computer Science,This article introduces a thin-film thermocoup...,A flexible film thermocouple temperature sensor
3,1511.06620,Use of Eigenvector Centrality to Detect Graph ...,Graph Isomorphism is one of the classical prob...,cs.SI cs.DM cs.DS,Computer Science,Graph Isomorphism is one of the classical prob...,Use of Eigenvector Centrality to Detect Graph ...
4,1801.09557,Families of Solutions of Algebraic Riccati Equ...,We consider Homogeneous Algebraic Riccati Equa...,math.OC,Mathematics,We consider Homogeneous Algebraic Riccati Equa...,Families of Solutions of Algebraic Riccati Equ...
...,...,...,...,...,...,...,...
499993,2106.07524,MIA-COV19D: COVID-19 Detection through 3-D Che...,Early and reliable COVID-19 diagnosis based on...,eess.IV cs.CV cs.LG,Machine Learning,Early and reliable COVID-19 diagnosis based on...,MIA-COV19D: COVID-19 Detection through 3-D Che...
499994,1607.07213,A transcendental approach to injectivity theor...,"In this paper, we study transcendental aspects...",math.CV math.AG math.DG,Mathematics,"In this paper, we study transcendental aspects...",A transcendental approach to injectivity theor...
499995,1512.08723,Emergent Weyl excitations in systems of polar ...,Weyl fermions are massless chiral particles fi...,cond-mat.quant-gas cond-mat.mes-hall cond-mat....,Computer Science,Weyl fermions are massless chiral particles fi...,Emergent Weyl excitations in systems of polar ...
499996,2202.11885,A Partition-and-Merge Algorithm for Solving th...,The Steiner tree problem aims to determine a m...,cs.DS,Computer Science,The Steiner tree problem aims to determine a m...,A Partition-and-Merge Algorithm for Solving th...


In [21]:
df_sampled.to_csv('training_data_papers.csv')